# CAS run on Colab (T4)
Runtime > Change runtime type > T4 GPU. Needs `pilot_v1.zip` and `metadata_with_conflicts.jsonl` (from 01b) on Drive. Re-running any cell after a disconnect resumes.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
DRIVE="/content/drive/MyDrive/sketchy_dataset_one_sketch"
STEPS=600          # first pass; raise to 3000 only if the loss/qualitative grid look sane
RUN=f"{DRIVE}/cas_runs/cas_v1"   # checkpoints live on Drive so they survive disconnects

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors peft scipy scikit-image
!rm -rf /content/repo && git clone -q -b claude/modest-ride-vkaas0 https://github.com/Harshitha-282007/constraint-driven-multimodal-diffusion /content/repo

In [ ]:
import glob, os, zipfile, shutil
L="/content/pilot_v1"
if not glob.glob(f"{L}/**/metadata.jsonl", recursive=True):
    os.makedirs(L, exist_ok=True); zipfile.ZipFile(f"{DRIVE}/pilot_v1.zip").extractall(L)
DATA=os.path.dirname(glob.glob(f"{L}/**/metadata.jsonl", recursive=True)[0])
shutil.copy(f"{DRIVE}/metadata_with_conflicts.jsonl", f"{DATA}/metadata_with_conflicts.jsonl")
print(DATA)

In [ ]:
!python /content/repo/governance/smoke_test.py 2>&1 | tail -3

In [ ]:
!python /content/repo/governance/train.py --data_dir "$DATA" --out "$RUN" --steps $STEPS --save_every 50

In [ ]:
!python /content/repo/governance/evaluate.py --data_dir "$DATA" --weights "$RUN/cas_weights.pt" --out "$RUN/eval" --n 12

In [ ]:
from IPython.display import Image, display
display(Image(f"{RUN}/eval/qualitative.png"))
import pandas as pd
print(pd.read_csv(f"{RUN}/eval/interventions_summary.csv").set_index("arm").T)